# NumPy 100 — Problems 81–90

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 81. Consider an array Z = [1,2,3,4,5,6,7,8,9,10,11,12,13,14], how to generate an array R = [[1,2,3,4], [2,3,4,5], [3,4,5,6], ..., [11,12,13,14]]? (★★★)

In [2]:
Z = np.arange(1, 15)

R = np.lib.stride_tricks.sliding_window_view(Z, 4)
print(R)
print(R.shape)                          # (11, 4)
print(R[0], R[-1])                      # [1 2 3 4] [11 12 13 14]

# older low-level way with as_strided (same result, easier to get wrong):
R2 = np.lib.stride_tricks.as_strided(Z, shape=(len(Z) - 3, 4), strides=(Z.strides[0], Z.strides[0]))
print(np.array_equal(R, R2))            # True

[[ 1  2  3  4]
 [ 2  3  4  5]
 [ 3  4  5  6]
 [ 4  5  6  7]
 [ 5  6  7  8]
 [ 6  7  8  9]
 [ 7  8  9 10]
 [ 8  9 10 11]
 [ 9 10 11 12]
 [10 11 12 13]
 [11 12 13 14]]
(11, 4)
[1 2 3 4] [11 12 13 14]
True


**Concept:** same idea as problem 76: a sliding window of length 4 over a vector of length 14 gives `14 - 4 + 1 = 11` rows, each shifted by one. `sliding_window_view` returns a read-only view (no copy); `as_strided` does it manually with strides.

**Needed help?** hint  
**Revisit on:** Nov 21

---
### 82. Compute a matrix rank (★★★)

In [3]:
rng = np.random.default_rng(42)

Z = rng.uniform(0, 1, (10, 10))
print(np.linalg.matrix_rank(Z))         # 10: a random square matrix is almost surely full rank

# a 6x6 matrix built from a 6x3 and a 3x6 matrix can have rank at most 3
A = rng.random((6, 3)) @ rng.random((3, 6))
print(np.linalg.matrix_rank(A))         # 3

# what matrix_rank does: count singular values that are not (numerically) zero
S = np.linalg.svd(A, compute_uv=False)
print(S.round(4))
print(np.sum(S > 1e-10))                # 3

10
3
[4.5718 0.2716 0.1353 0.     0.     0.    ]
3


**Concept:** the rank is the number of linearly independent rows (or columns). `np.linalg.matrix_rank` computes it from the singular values (`np.linalg.svd`): it counts how many are above a small tolerance. A product of a 6×3 and a 3×6 matrix can't have rank above 3, which the three non-zero singular values confirm.

**Needed help?** hint  
**Revisit on:** Nov 21

---
### 83. How to find the most frequent value in an array?

In [4]:
rng = np.random.default_rng(42)
Z = rng.integers(0, 10, 50)

print(np.bincount(Z).argmax())          # fast; only for non-negative integers

# general version, works for any values:
values, counts = np.unique(Z, return_counts=True)
print(values[counts.argmax()])
print(dict(zip(values.tolist(), counts.tolist())))

7
7
{0: 6, 1: 3, 2: 3, 3: 2, 4: 7, 5: 5, 6: 5, 7: 9, 8: 7, 9: 3}


**Concept:** `np.bincount(Z).argmax()` counts each integer and returns the most frequent one, but it only works for non-negative integers. `np.unique(Z, return_counts=True)` works for any values: pick the one with the largest count. In a tie, `argmax` returns the first, which is the smallest value.

**Needed help?** no  
**Revisit on:** not needed

---
### 84. Extract all the contiguous 3x3 blocks from a random 10x10 matrix (★★★)

In [5]:
rng = np.random.default_rng(42)
Z = rng.integers(0, 5, (10, 10))

C = np.lib.stride_tricks.sliding_window_view(Z, (3, 3))
print(C.shape)                          # (8, 8, 3, 3): an 8x8 grid of 3x3 blocks
print(np.array_equal(C[2, 5], Z[2:5, 5:8]))     # True: block (2, 5) is the slice starting there

(8, 8, 3, 3)
True


**Concept:** `sliding_window_view(Z, (3, 3))` returns every contiguous 3×3 block as a view of shape `(8, 8, 3, 3)`: `10 - 3 + 1 = 8` positions along each axis. Block `[i, j]` equals `Z[i:i+3, j:j+3]`. It's read-only; use `.copy()` if you need to modify blocks. The original used `as_strided`, which does the same by hand.

**Needed help?** solution  
**Revisit on:** Nov 21

---
### 85. Create a 2D array subclass such that Z[i,j] == Z[j,i] (★★★)

In [6]:
class Symmetric(np.ndarray):
    def __setitem__(self, index, value):
        i, j = index
        super().__setitem__((i, j), value)
        super().__setitem__((j, i), value)      # mirror every write

def symmetric(Z):
    return np.asarray(Z + Z.T - np.diag(Z.diagonal())).view(Symmetric)

rng = np.random.default_rng(42)
S = symmetric(rng.integers(0, 10, (5, 5)))
S[2, 3] = 42
print(S)
print(S[2, 3], S[3, 2])                 # 42 42
print(np.array_equal(S, S.T))           # True

[[ 0 15 11 11  9]
 [15  0 15  7  3]
 [11 15  7 42  8]
 [11  7 42  8 13]
 [ 9  3  8 13  7]]
42 42
True


**Concept:** subclass `np.ndarray` and override `__setitem__` so every write to `[i, j]` also writes `[j, i]`. The helper `Z + Z.T - diag(diag(Z))` first turns any square matrix into a symmetric one. This version only supports writing with an integer pair `S[i, j] = v`, not slices.

**Needed help?** solution  
**Revisit on:** Nov 21

---
### 86. Consider a set of p matrices with shape (n,n) and a set of p vectors with shape (n,1). How to compute the sum of of the p matrix products at once? (result has shape (n,1)) (★★★)

In [7]:
rng = np.random.default_rng(42)
p, n = 10, 5
M = rng.random((p, n, n))               # p matrices (n, n)
V = rng.random((p, n, 1))               # p vectors (n, 1)

S = np.tensordot(M, V, axes=[[0, 2], [0, 1]])
print(S.shape)                          # (5, 1)

# the same with einsum, and with a plain loop as a check
S2 = np.einsum("pij,pjk->ik", M, V)
S3 = sum(M[k] @ V[k] for k in range(p))
print(np.allclose(S, S2), np.allclose(S, S3))   # True True

(5, 1)
True True


**Concept:** summing `p` matrix-vector products is a contraction over two axes at once: the matrix index `j` and the sample index `p`. `np.tensordot(M, V, axes=[[0, 2], [0, 1]])` sums over axis 0 and 2 of `M` against axis 0 and 1 of `V`. `np.einsum("pij,pjk->ik", M, V)` says the same thing in index notation, which is often easier to read.

**Needed help?** solution  
**Revisit on:** Nov 21

---
### 87. Consider a 16x16 array, how to get the block-sum (block size is 4x4)? (★★★)

In [8]:
rng = np.random.default_rng(42)
Z = rng.integers(0, 10, (16, 16))
k = 4

S = np.add.reduceat(np.add.reduceat(Z, np.arange(0, Z.shape[0], k), axis=0),
                    np.arange(0, Z.shape[1], k), axis=1)
print(S.shape)                          # (4, 4)

# simpler when the size divides evenly: reshape into blocks and sum
S2 = Z.reshape(4, k, 4, k).sum(axis=(1, 3))
print(np.array_equal(S, S2))            # True
print(S[0, 0] == Z[:4, :4].sum())       # True

(4, 4)
True
True


**Concept:** `np.add.reduceat(Z, starts, axis)` sums slices that begin at each index in `starts`; doing it along both axes gives block sums. When the block size divides the array evenly, reshape to `(4, k, 4, k)` and sum over axes 1 and 3, which is shorter and clearer. Replace `sum` with `mean` for block averages (downsampling an image).

**Needed help?** solution  
**Revisit on:** Nov 21

---
### 88. How to implement the Game of Life using numpy arrays? (★★★)

In [9]:
def iterate(Z):
    # count the 8 neighbours of every interior cell
    N = (Z[0:-2, 0:-2] + Z[0:-2, 1:-1] + Z[0:-2, 2:] +
         Z[1:-1, 0:-2]                 + Z[1:-1, 2:] +
         Z[2:  , 0:-2] + Z[2:  , 1:-1] + Z[2:  , 2:])

    birth = (N == 3) & (Z[1:-1, 1:-1] == 0)
    survive = ((N == 2) | (N == 3)) & (Z[1:-1, 1:-1] == 1)

    Z[...] = 0                          # border cells stay dead
    Z[1:-1, 1:-1][birth | survive] = 1
    return Z

def show(Z):
    print("\n".join("".join("#" if c else "." for c in row) for row in Z))

# test 1: a glider moves one cell down-right every 4 generations
glider = np.array([[0, 1, 0], [0, 0, 1], [1, 1, 1]])
G = np.zeros((8, 8), dtype=int)
G[1:4, 1:4] = glider
show(G)
for _ in range(4):
    G = iterate(G)
print()
show(G)

expected = np.zeros((8, 8), dtype=int)
expected[2:5, 2:5] = glider
print(np.array_equal(G, expected))      # True

# test 2: a blinker flips between horizontal and vertical
B = np.zeros((5, 5), dtype=int)
B[2, 1:4] = 1
B = iterate(B)
print(B[1:4, 2])                        # [1 1 1] -> now vertical

# random start, 100 generations
rng = np.random.default_rng(42)
Z = rng.integers(0, 2, (50, 50))
for _ in range(100):
    Z = iterate(Z)
print(Z.sum(), "cells alive")

........
..#.....
...#....
.###....
........
........
........
........

........
........
...#....
....#...
..###...
........
........
........
True
[1 1 1]
209 cells alive


**Concept:** the rules are applied to the whole grid at once. Add eight shifted views of the array to get every cell's neighbour count `N`, then use boolean masks: a dead cell with 3 neighbours is born; a live cell with 2 or 3 neighbours survives; everything else dies. The border is treated as always dead. A glider and a blinker are classic tests, since their behavior is known.

**Needed help?** solution  
**Revisit on:** Nov 21

---
### 89. How to get the n largest values of an array (★★★)

In [10]:
rng = np.random.default_rng(42)
Z = rng.permutation(10000)              # 0..9999 in random order
n = 5

# simple: sort everything, take the last n
print(Z[np.argsort(Z)[-n:]])

# faster for big arrays: partial sort (the n largest, in no particular order)
top = Z[np.argpartition(-Z, n)[:n]]
print(np.sort(top)[::-1])               # [9999 9998 9997 9996 9995]

print(np.partition(Z, -n)[-n:])         # values only, unordered

[9995 9996 9997 9998 9999]
[9999 9998 9997 9996 9995]
[9995 9996 9997 9998 9999]


**Concept:** `np.argsort(Z)[-n:]` sorts the whole array, which costs `O(N log N)`. `np.argpartition(-Z, n)[:n]` only guarantees the `n` largest end up in the first `n` slots (unordered), which is about `O(N)`, so it's much faster for large arrays. Sort the small result afterwards if you need order. `np.partition(Z, -n)[-n:]` gives values instead of indices.

**Needed help?** hint  
**Revisit on:** Nov 21

---
### 90. Given an arbitrary number of vectors, build the cartesian product (every combination of every item) (★★★)

In [11]:
def cartesian(*arrays):
    grids = np.meshgrid(*arrays, indexing="ij")
    return np.stack(grids, axis=-1).reshape(-1, len(arrays))

result = cartesian([1, 2, 3], [4, 5], [6, 7])
print(result)
print(result.shape)                     # (12, 3) = 3 * 2 * 2 combinations

# check against the standard library
import itertools
expected = np.array(list(itertools.product([1, 2, 3], [4, 5], [6, 7])))
print(np.array_equal(result, expected))     # True

[[1 4 6]
 [1 4 7]
 [1 5 6]
 [1 5 7]
 [2 4 6]
 [2 4 7]
 [2 5 6]
 [2 5 7]
 [3 4 6]
 [3 4 7]
 [3 5 6]
 [3 5 7]]
(12, 3)
True


**Concept:** `np.meshgrid(*arrays, indexing="ij")` makes one grid per input vector, holding every combination. Stack them along a new last axis and reshape to `(-1, number_of_vectors)` so each row is one combination, in the same order as `itertools.product`. The row count is the product of the input lengths, so it grows fast.

**Needed help?** solution  
**Revisit on:** Nov 21